# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Terra Jiang (yjiang66@berkeley.edu)

### Notebook Structure:

1. Notebook Imports: 
    * Basic and necessary libraries.
    * Datasets: `results/train.csv`, `results/val.csv`, and `results/test.csv` from the data processing step.
2. Model Developments:
    * Random Forest model: 
        * Model development with training and validation datasets.
        * Model evaluation with training and testing datasets.
    * XGBoost model: 
        * Model development with training and validation datasets.
        * Model evaluation with training and testing datasets.
    * Model Comparison: `NotImplemented`
3. Notebook Exports: 
    * Export models: `results/rf.pkl` representing the Random Forest model and `results/xgb.pkl` representing the XGBoost model. 
    * Export evaluation results: `results/stats_rf_xgb.csv`

## 1. Notebook Imports

### 1.1 Import basic and necessary libraries:

In [ ]:
# Basic imports
import pandas as pd
from matplotlib import pyplot as plt
import joblib

# For Random Forest and XGBoost models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# For model evaluations
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Compress all warnings
import warnings

warnings.filterwarnings("ignore")


### 1.2 Import datasets:

In [ ]:
# Training dataset
df_train = pd.read_csv("../results/train.csv")
X_train = df_train.drop(columns=["readmitted"], axis=1)
Y_train = df_train["readmitted"]
# Validation dataset
df_val = pd.read_csv("../results/val.csv")
X_val = df_val.drop(columns=["readmitted"], axis=1)
Y_val = df_val["readmitted"]
# Testing dataset
df_test = pd.read_csv("../results/test.csv")
X_test = df_test.drop(columns=["readmitted"], axis=1)
Y_test = df_test["readmitted"]

In [ ]:
# Display top 5 rows of the training dataset
display(df_train.head())

In [ ]:
# Check the shape of all variables
print(f"The shape of X_train is: {X_train.shape}")
print(f"The shape of Y_train is: {Y_train.shape}")
print(f"The shape of X_val is: {X_val.shape}")
print(f"The shape of Y_val is: {Y_val.shape}")
print(f"The shape of X_test is: {X_test.shape}")
print(f"The shape of Y_test is: {Y_test.shape}")

## 2. Model Developments

### 2.1 Random Forest Model

In [ ]:
# Make copies from X&Y datasets
X_train_rf = X_train.copy()
Y_train_rf = Y_train.copy()
X_val_rf = X_val.copy()
Y_val_rf = Y_val.copy()
X_test_rf = X_test.copy()
Y_test_rf = Y_test.copy()

# Print the shapes for each dataset
print("(rf) Shape of X_train: ", X_train_rf.shape)
print("(rf) Shape of Y_train: ", Y_train_rf.shape)
print("(rf) Shape of X_val: ", X_val_rf.shape)
print("(rf) Shape of Y_val: ", Y_val_rf.shape)
print("(rf) Shape of X_test: ", X_test_rf.shape)
print("(rf) Shape of Y_test: ", Y_test_rf.shape)

#### 2.1.1 Model development with training and validation datasets

In [ ]:
# Hyperparameter Tuning
tuning_options_rf = {
    "n_estimators": [100, 200, 300],  # Number of trees
    "max_depth": [6, 8, 12, 16],  # Maximum depth of each tree
    "min_samples_split": [2, 5, 10],
}  # Min samples to split for each step

# Create and fit the model on the training dataset
model_rf = RandomizedSearchCV(
    RandomForestClassifier(),  # Estimator - Random Forest
    param_distributions=tuning_options_rf,
    n_iter=30,  # Number of parameter settings sampled
    n_jobs=-1,  # Optimize CPU
    scoring="f1",
)  # Evaluation metric

clf_rf = model_rf.fit(X_train_rf, Y_train_rf)
print(f"Accuracy on the training dataset: {clf_rf.score(X_train_rf, Y_train_rf):.2%}")
print(f"Accuracy on the validation dataset: {clf_rf.score(X_val_rf, Y_val_rf):.2%}")

# Obtain the top 5 importance feature names and their importance scores
print("\nTop 5 important features:")
print(
    pd.DataFrame(
        {
            "features": X_train_rf.columns,
            "importance": clf_rf.best_estimator_.feature_importances_,
        }
    )
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
    .head()
)

#### 2.1.2 Model evaluation with training and testing datasets

In [ ]:
# Calculate the predicted values
Y_train_hat_rf = model_rf.predict(X_train_rf)
Y_test_hat_rf = model_rf.predict(X_test_rf)

# Calculate accuracy values on both train and test datasets
accuracy_train_rf = accuracy_score(Y_train_rf, Y_train_hat_rf)
accuracy_test_rf = accuracy_score(Y_test_rf, Y_test_hat_rf)
print(f"\nThe aggregate accuracy from the training dataset is: {accuracy_train_rf:.2%}")
print(
    f"The aggregate accuracy from the testing dataset is: {accuracy_test_rf:.2%}",
    end="\n\n",
)
print(
    classification_report(
        Y_test_rf, Y_test_hat_rf, target_names=["No readmission", "Readmission"]
    )
)

# Plot the confusion matrix
cm_rf = confusion_matrix(Y_test_rf, Y_test_hat_rf)
disp = ConfusionMatrixDisplay(cm_rf, display_labels=["No readmission", "Readmitted"])
disp.plot()
plt.grid(False)  # Hide the white grid lines
plt.title("Confusion Matrix on The Test Dataset")
plt.show()

# Evaluation from the confusion matrix
precision_rf = cm_rf[1][1] / cm_rf[:, 1].sum()
recall_rf = cm_rf[1][1] / cm_rf[1].sum()
print(f"Precisions: {precision_rf:.2%}")
print(f"False Positives: {cm_rf[0][1]}")
print(f"Recalls: {recall_rf:.3%}")
print(f"False Negatives: {cm_rf[1][0]}")
print(f"\nF1 score: {2 * precision_rf * recall_rf / (precision_rf + recall_rf):.2%}")

### 2.2 XGBoost Model

In [ ]:
# Make copies from X&Y datasets
X_train_xgb = X_train.copy()
Y_train_xgb = Y_train.copy()
X_val_xgb = X_val.copy()
Y_val_xgb = Y_val.copy()
X_test_xgb = X_test.copy()
Y_test_xgb = Y_test.copy()

# Print the shapes for each dataset
print("(xgb) Shape of X_train: ", X_train_xgb.shape)
print("(xgb) Shape of Y_train: ", Y_train_xgb.shape)
print("(xgb) Shape of X_val: ", X_val_xgb.shape)
print("(xgb) Shape of Y_val: ", Y_val_xgb.shape)
print("(xgb) Shape of X_test: ", X_test_xgb.shape)
print("(xgb) Shape of Y_test: ", Y_test_xgb.shape)

#### 2.2.1 Model development with training and validation datasets

In [ ]:
# Hyperparameter Tuning
tuning_options_xgb = {
    "n_estimators": [30, 50, 100, 200, 300, 500],  # Number of boosting rounds
    "max_depth": [3, 6, 8, 12, 16],  # Maximum depth of each tree
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.4],  # Step size shrinkage
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3],
    "min_child_weight": [1, 3, 5, 7],
}

# Create and fit the model on the training dataset
model_xgb = RandomizedSearchCV(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        use_label_encoder=False,
        scale_pos_weight=len(Y_train[Y_train == 0]) / len(Y_train[Y_train == 1]),
    ),  # Estimator - XGBoost
    param_distributions=tuning_options_xgb,
    n_iter=30,  # Number of parameter settings sampled
    n_jobs=-1,  # Optimize CPU
    scoring="f1",
)  # Evaluation metric


clf_xgb = model_xgb.fit(X_train_xgb, Y_train_xgb)
print(
    f"Accuracy on the training dataset: {clf_xgb.score(X_train_xgb, Y_train_xgb):.2%}"
)
print(f"Accuracy on the validation dataset: {clf_xgb.score(X_val_xgb, Y_val_xgb):.2%}")

# Obtain the top 5 importance feature names and their importance scores
print("\nTop 5 important features:")
print(
    pd.DataFrame(
        {
            "features": X_train_xgb.columns,
            "importance": clf_xgb.best_estimator_.feature_importances_,
        }
    )
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
    .head()
)

#### 2.2.2 Model evaluation with training and testing datasets

In [ ]:
# Calculate the predicted values
Y_train_hat_xgb = model_xgb.predict(X_train_xgb)
Y_test_hat_xgb = model_xgb.predict(X_test_xgb)

# Calculate accuracy values on both train and test datasets
accuracy_train_xgb = accuracy_score(Y_train_xgb, Y_train_hat_xgb)
accuracy_test_xgb = accuracy_score(Y_test_xgb, Y_test_hat_xgb)
print(
    f"\nThe aggregate accuracy from the training dataset is: {accuracy_train_xgb:.2%}"
)
print(
    f"The aggregate accuracy from the testing dataset is: {accuracy_test_xgb:.2%}",
    end="\n\n",
)
print(
    classification_report(
        Y_test_xgb, Y_test_hat_xgb, target_names=["No readmission", "Readmission"]
    )
)

# Plot the confusion matrix
cm_xgb = confusion_matrix(Y_test_xgb, Y_test_hat_xgb)
disp = ConfusionMatrixDisplay(cm_xgb, display_labels=["No readmission", "Readmitted"])
disp.plot()
plt.grid(False)  # Hide the white grid lines
plt.title("Confusion Matrix on The Test Dataset")
plt.show()

# Evaluation from the confusion matrix
precision_xgb = cm_xgb[1][1] / cm_xgb[:, 1].sum()
recall_xgb = cm_xgb[1][1] / cm_xgb[1].sum()
print(f"Precisions: {precision_xgb:.2%}")
print(f"False Positives: {cm_xgb[0][1]}")
print(f"Recalls: {recall_xgb:.3%}")
print(f"False Negatives: {cm_xgb[1][0]}")
print(
    f"\nF1 score: {2 * precision_xgb * recall_xgb / (precision_xgb + recall_xgb):.2%}"
)

## 3. Notebook Exports

#### 3.1 Export models

In [ ]:
# Export the random forest and xgboost models into results/
joblib.dump(model_rf, "../results/rf.pkl")
joblib.dump(model_xgb, "../results/xgb.pkl")

#### 3.2 Export evaluation results

In [ ]:
# Initialize the dataframe
stats = pd.DataFrame(
    columns=[
        "feature_importance",
        "precision",
        "recall",
        "false_positive",
        "false_negative",
        "final_accuracy_train",
        "final_accuracy_test",
        "f1",
    ]
)

# Insert model statistics from random forest and xgboost
stats_rf = pd.DataFrame(
    [
        {
            "feature_importance": clf_rf.best_estimator_.feature_importances_,  # Force importance values to be included in one row
            "precision": precision_rf,
            "recall": recall_rf,
            "false_positive": cm_rf[0][1],
            "false_negative": cm_rf[1][0],
            "final_accuracy_train": accuracy_score(Y_train_rf, Y_train_hat_rf),
            "final_accuracy_test": accuracy_score(Y_test_rf, Y_test_hat_rf),
            "f1": 2 * precision_rf * recall_rf / (precision_rf + recall_rf),
        }
    ]
)
stats_xgb = pd.DataFrame(
    [
        {
            "feature_importance": clf_xgb.best_estimator_.feature_importances_,  # Force importance values to be included in one row
            "precision": precision_xgb,
            "recall": recall_xgb,
            "false_positive": cm_xgb[0][1],
            "false_negative": cm_xgb[1][0],
            "final_accuracy_train": accuracy_score(Y_train_xgb, Y_train_hat_xgb),
            "final_accuracy_test": accuracy_score(Y_test_xgb, Y_test_hat_xgb),
            "f1": 2 * precision_xgb * recall_xgb / (precision_xgb + recall_xgb),
        }
    ]
)

# Combine the stats and export as csv
stats = pd.concat([stats, stats_rf, stats_xgb], ignore_index=True)
stats.to_csv("../results/stats_rf_xgb.csv", index=False)